# Healthcare Claims Fraud & Anomaly Assistant

## Business (healthcare claims) Challenge

This notebook is about a healthcare claims project， this assistant can look up:

- claim details
- provider risk profile
- patient utilization
- anomalous claims

The goal is to demonstrate:

```text
LLM + Python tools + SQLite database = Healthcare AI assistant
```

## 1. Import libraries

In [1]:
# If needed, install packages first:
# !pip install openai gradio python-dotenv truststore certifi

import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

try:
    import truststore
    truststore.inject_into_ssl()
    print("truststore loaded successfully")
except Exception as e:
    print("truststore not loaded:", e)

c:\Users\Sealion\anaconda3\envs\medical-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


truststore loaded successfully


## 2. Set up OpenAI

In [2]:
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("OPENAI_API_KEY found")
else:
    print("OPENAI_API_KEY not found. Please add it to your .env file.")

openai = OpenAI()
MODEL = "gpt-4o-mini"

print("Model:", MODEL)

OPENAI_API_KEY found
Model: gpt-4o-mini


## 3. System message

This tells the model how to behave.

In [3]:
system_message = """
You are a helpful healthcare claims fraud and anomaly detection assistant.

You help users understand healthcare claims, provider behavior, patient utilization,
billing anomalies, and possible fraud, waste, or abuse.

Give clear and concise answers.

When claim, provider, or patient data is needed, use the available tools.

Do not make a final fraud accusation. Use cautious language such as:
- possible anomaly
- suspicious pattern
- needs further review
- may require investigation

Always explain the reason for the risk score or anomaly flag.
"""

## 4. Create a small SQLite database

The database has 3 tables:

1. `claims`
2. `providers`
3. `patients`

In [4]:
DB = "healthcare_claims.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()

    cursor.execute("DROP TABLE IF EXISTS claims")
    cursor.execute("DROP TABLE IF EXISTS providers")
    cursor.execute("DROP TABLE IF EXISTS patients")

    cursor.execute("""
        CREATE TABLE claims (
            claim_id TEXT PRIMARY KEY,
            patient_id TEXT,
            provider_id TEXT,
            diagnosis_code TEXT,
            procedure_code TEXT,
            claim_amount REAL,
            paid_amount REAL,
            service_date TEXT,
            place_of_service TEXT,
            claim_type TEXT,
            anomaly_flag INTEGER,
            anomaly_reason TEXT
        )
    """)

    cursor.execute("""
        CREATE TABLE providers (
            provider_id TEXT PRIMARY KEY,
            provider_name TEXT,
            specialty TEXT,
            state TEXT,
            total_claims INTEGER,
            avg_claim_amount REAL,
            risk_score REAL,
            risk_reason TEXT
        )
    """)

    cursor.execute("""
        CREATE TABLE patients (
            patient_id TEXT PRIMARY KEY,
            age INTEGER,
            state TEXT,
            total_claims INTEGER,
            total_paid REAL,
            utilization_level TEXT
        )
    """)

    conn.commit()

print("Database created:", DB)

Database created: healthcare_claims.db


## 5. Insert sample data

These are fake examples for learning only.

In [5]:
sample_claims = [
    ("C1001", "P001", "PR001", "E11.9", "99213", 180.00, 120.00, "2026-01-10", "Office", "Professional", 0, "Normal office visit"),
    ("C1002", "P002", "PR002", "M54.5", "72148", 2400.00, 1800.00, "2026-01-11", "Imaging Center", "Imaging", 1, "High imaging cost compared with common peer pattern"),
    ("C1003", "P003", "PR003", "R05.9", "99214", 300.00, 210.00, "2026-01-12", "Office", "Professional", 0, "Moderate office visit"),
    ("C1004", "P004", "PR004", "Z79.899", "A0428", 1500.00, 1300.00, "2026-01-12", "Ambulance", "Transportation", 1, "Possible medical necessity review needed"),
    ("C1005", "P005", "PR005", "E78.5", "80061", 950.00, 700.00, "2026-01-13", "Laboratory", "Laboratory", 1, "Unusually high lab billing amount"),
    ("C1006", "P001", "PR002", "M54.5", "72148", 2450.00, 1850.00, "2026-01-14", "Imaging Center", "Imaging", 1, "Repeated high-cost imaging pattern")
]

sample_providers = [
    ("PR001", "Northside Family Clinic", "Family Medicine", "GA", 450, 165.00, 0.12, "Low risk: billing close to peer average"),
    ("PR002", "Advanced Imaging Group", "Radiology", "GA", 1800, 2350.00, 0.82, "High risk: repeated high-cost imaging claims"),
    ("PR003", "Peachtree Primary Care", "Internal Medicine", "GA", 520, 220.00, 0.20, "Low to moderate risk"),
    ("PR004", "Rapid Medical Transport", "Ambulance", "GA", 980, 1450.00, 0.76, "Elevated risk: medical necessity review often needed"),
    ("PR005", "Metro Diagnostic Lab", "Laboratory", "GA", 2200, 890.00, 0.88, "High risk: excessive lab billing pattern")
]

sample_patients = [
    ("P001", 67, "GA", 18, 9500.00, "High"),
    ("P002", 72, "GA", 7, 3900.00, "Moderate"),
    ("P003", 58, "GA", 4, 900.00, "Low"),
    ("P004", 80, "GA", 11, 6800.00, "High"),
    ("P005", 63, "GA", 9, 4200.00, "Moderate")
]

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.executemany("INSERT INTO claims VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", sample_claims)
    cursor.executemany("INSERT INTO providers VALUES (?, ?, ?, ?, ?, ?, ?, ?)", sample_providers)
    cursor.executemany("INSERT INTO patients VALUES (?, ?, ?, ?, ?, ?)", sample_patients)
    conn.commit()

print("Sample data inserted.")

Sample data inserted.


## 6. Tool 1: Get claim details

In [6]:
def get_claim_details(claim_id):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            """
            SELECT claim_id, patient_id, provider_id, diagnosis_code, procedure_code,
                   claim_amount, paid_amount, service_date, place_of_service,
                   claim_type, anomaly_flag, anomaly_reason
            FROM claims
            WHERE claim_id = ?
            """,
            (claim_id,)
        )
        row = cursor.fetchone()

    if not row:
        return f"No claim found for claim_id {claim_id}."

    claim = {
        "claim_id": row[0],
        "patient_id": row[1],
        "provider_id": row[2],
        "diagnosis_code": row[3],
        "procedure_code": row[4],
        "claim_amount": row[5],
        "paid_amount": row[6],
        "service_date": row[7],
        "place_of_service": row[8],
        "claim_type": row[9],
        "anomaly_flag": bool(row[10]),
        "anomaly_reason": row[11]
    }

    return json.dumps(claim, indent=2)

In [7]:
print(get_claim_details("C1002"))

{
  "claim_id": "C1002",
  "patient_id": "P002",
  "provider_id": "PR002",
  "diagnosis_code": "M54.5",
  "procedure_code": "72148",
  "claim_amount": 2400.0,
  "paid_amount": 1800.0,
  "service_date": "2026-01-11",
  "place_of_service": "Imaging Center",
  "claim_type": "Imaging",
  "anomaly_flag": true,
  "anomaly_reason": "High imaging cost compared with common peer pattern"
}


## 7. Tool 2: Get provider risk profile

In [8]:
def get_provider_risk(provider_id):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            """
            SELECT provider_id, provider_name, specialty, state,
                   total_claims, avg_claim_amount, risk_score, risk_reason
            FROM providers
            WHERE provider_id = ?
            """,
            (provider_id,)
        )
        row = cursor.fetchone()

    if not row:
        return f"No provider found for provider_id {provider_id}."

    provider = {
        "provider_id": row[0],
        "provider_name": row[1],
        "specialty": row[2],
        "state": row[3],
        "total_claims": row[4],
        "avg_claim_amount": row[5],
        "risk_score": row[6],
        "risk_reason": row[7]
    }

    return json.dumps(provider, indent=2)

In [9]:
print(get_provider_risk("PR002"))

{
  "provider_id": "PR002",
  "provider_name": "Advanced Imaging Group",
  "specialty": "Radiology",
  "state": "GA",
  "total_claims": 1800,
  "avg_claim_amount": 2350.0,
  "risk_score": 0.82,
  "risk_reason": "High risk: repeated high-cost imaging claims"
}


## 8. Tool 3: Get patient utilization

In [10]:
def get_patient_utilization(patient_id):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            """
            SELECT patient_id, age, state, total_claims, total_paid, utilization_level
            FROM patients
            WHERE patient_id = ?
            """,
            (patient_id,)
        )
        row = cursor.fetchone()

    if not row:
        return f"No patient found for patient_id {patient_id}."

    patient = {
        "patient_id": row[0],
        "age": row[1],
        "state": row[2],
        "total_claims": row[3],
        "total_paid": row[4],
        "utilization_level": row[5]
    }

    return json.dumps(patient, indent=2)

In [11]:
print(get_patient_utilization("P001"))

{
  "patient_id": "P001",
  "age": 67,
  "state": "GA",
  "total_claims": 18,
  "total_paid": 9500.0,
  "utilization_level": "High"
}


## 9. Tool 4: Find anomalous claims

In [12]:
def find_anomalous_claims(limit=5):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            """
            SELECT claim_id, patient_id, provider_id, procedure_code,
                   claim_amount, paid_amount, claim_type, anomaly_reason
            FROM claims
            WHERE anomaly_flag = 1
            LIMIT ?
            """,
            (limit,)
        )
        rows = cursor.fetchall()

    if not rows:
        return "No anomalous claims found."

    claims = []

    for row in rows:
        claims.append({
            "claim_id": row[0],
            "patient_id": row[1],
            "provider_id": row[2],
            "procedure_code": row[3],
            "claim_amount": row[4],
            "paid_amount": row[5],
            "claim_type": row[6],
            "anomaly_reason": row[7]
        })

    return json.dumps(claims, indent=2)

In [13]:
print(find_anomalous_claims(3))

[
  {
    "claim_id": "C1002",
    "patient_id": "P002",
    "provider_id": "PR002",
    "procedure_code": "72148",
    "claim_amount": 2400.0,
    "paid_amount": 1800.0,
    "claim_type": "Imaging",
    "anomaly_reason": "High imaging cost compared with common peer pattern"
  },
  {
    "claim_id": "C1004",
    "patient_id": "P004",
    "provider_id": "PR004",
    "procedure_code": "A0428",
    "claim_amount": 1500.0,
    "paid_amount": 1300.0,
    "claim_type": "Transportation",
    "anomaly_reason": "Possible medical necessity review needed"
  },
  {
    "claim_id": "C1005",
    "patient_id": "P005",
    "provider_id": "PR005",
    "procedure_code": "80061",
    "claim_amount": 950.0,
    "paid_amount": 700.0,
    "claim_type": "Laboratory",
    "anomaly_reason": "Unusually high lab billing amount"
  }
]


## 10. Describe tools to OpenAI

In [14]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_claim_details",
            "description": "Get detailed information for a healthcare claim using a claim_id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "claim_id": {
                        "type": "string",
                        "description": "The healthcare claim ID, for example C1002."
                    }
                },
                "required": ["claim_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_provider_risk",
            "description": "Get the provider risk profile using a provider_id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "provider_id": {
                        "type": "string",
                        "description": "The provider ID, for example PR002."
                    }
                },
                "required": ["provider_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_patient_utilization",
            "description": "Get patient utilization information using a patient_id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "patient_id": {
                        "type": "string",
                        "description": "The patient ID, for example P001."
                    }
                },
                "required": ["patient_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "find_anomalous_claims",
            "description": "Find healthcare claims flagged as anomalous or suspicious.",
            "parameters": {
                "type": "object",
                "properties": {
                    "limit": {
                        "type": "integer",
                        "description": "Maximum number of anomalous claims to return."
                    }
                },
                "required": ["limit"],
                "additionalProperties": False
            }
        }
    }
]

print("Number of tools:", len(tools))

Number of tools: 4


## 11. Handle tool calls

In [15]:
def handle_tool_calls(message):
    tool_responses = []

    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        print("Tool called:", function_name)
        print("Arguments:", arguments)

        if function_name == "get_claim_details":
            result = get_claim_details(arguments.get("claim_id"))

        elif function_name == "get_provider_risk":
            result = get_provider_risk(arguments.get("provider_id"))

        elif function_name == "get_patient_utilization":
            result = get_patient_utilization(arguments.get("patient_id"))

        elif function_name == "find_anomalous_claims":
            result = find_anomalous_claims(arguments.get("limit", 5))

        else:
            result = f"Unknown tool: {function_name}"

        tool_responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })

    return tool_responses

## 12. Chat function with tool calling

This version works with the older Gradio history format:

```python
history = [
    ["user message", "assistant response"]
]
```

In [16]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}]

    for item in history:
        if isinstance(item, dict):
            messages.append({
                "role": item["role"],
                "content": item["content"]
            })
        else:
            user_message, assistant_message = item
            messages.append({"role": "user", "content": user_message})
            messages.append({"role": "assistant", "content": assistant_message})

    messages.append({"role": "user", "content": message})

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        tool_request_message = response.choices[0].message
        tool_response_messages = handle_tool_calls(tool_request_message)

        messages.append(tool_request_message)
        messages.extend(tool_response_messages)

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

## 13. Manual tests

In [17]:
history = []

answer = chat("What is claim C1002? Is it suspicious?", history)

print(answer)

Tool called: get_claim_details
Arguments: {'claim_id': 'C1002'}
Tool called: get_provider_risk
Arguments: {'provider_id': 'PR002'}
Tool called: get_patient_utilization
Arguments: {'patient_id': 'P002'}
Claim C1002 is an imaging claim submitted by the provider "Advanced Imaging Group" for a patient with the following details:

- **Claim ID**: C1002
- **Patient ID**: P002
- **Provider ID**: PR002 (Advanced Imaging Group)
- **Diagnosis Code**: M54.5
- **Procedure Code**: 72148
- **Claim Amount**: $2,400
- **Paid Amount**: $1,800
- **Service Date**: January 11, 2026
- **Place of Service**: Imaging Center
- **Claim Type**: Imaging
- **Anomaly Flag**: Yes
- **Anomaly Reason**: High imaging cost compared with common peer pattern

### Suspicion Assessment
This claim has been flagged as anomalous due to its high cost relative to what similar claims typically incur. 

Additionally, the provider, Advanced Imaging Group, has a high-risk score of 0.82. This risk score is attributed to a pattern of 

In [18]:
history = [
    ["Hello", "Hi! I can help review healthcare claims and provider risk."]
]

answer = chat("What is the risk profile for provider PR002?", history)

print(answer)

Tool called: get_provider_risk
Arguments: {'provider_id': 'PR002'}
The risk profile for provider **PR002** (Advanced Imaging Group) is as follows:

- **Specialty**: Radiology
- **Location**: Georgia (GA)
- **Total Claims**: 1,800
- **Average Claim Amount**: $2,350

**Risk Score**: 0.82  
**Risk Reason**: The provider is considered high risk due to repeated high-cost imaging claims.

This suggests a possible pattern of billing that may require further review to ensure it aligns with standard practices.


In [19]:
history = []

answer = chat("Show me 3 anomalous claims and explain why they may need review.", history)

print(answer)

Tool called: find_anomalous_claims
Arguments: {'limit': 3}
Here are three anomalous claims that may require further review, along with the reasons for their flags:

1. **Claim ID: C1002**
   - **Patient ID:** P002
   - **Provider ID:** PR002
   - **Procedure Code:** 72148
   - **Claim Amount:** $2,400
   - **Paid Amount:** $1,800
   - **Claim Type:** Imaging
   - **Anomaly Reason:** High imaging cost compared with common peer pattern.
   
   **Possible Review Reason:** This claim's imaging cost is significantly higher than what is typically observed among peers for similar services, which raises questions about the necessity or appropriateness of the imaging performed.

2. **Claim ID: C1004**
   - **Patient ID:** P004
   - **Provider ID:** PR004
   - **Procedure Code:** A0428
   - **Claim Amount:** $1,500
   - **Paid Amount:** $1,300
   - **Claim Type:** Transportation
   - **Anomaly Reason:** Possible medical necessity review needed.
   
   **Possible Review Reason:** This transportat

## 14. Launch Gradio chatbot

Do not use `type="messages"` if your Gradio version does not support it.

In [20]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


c:\Users\Sealion\anaconda3\envs\medical-ai\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called: get_claim_details
Arguments: {'claim_id': 'C1002'}


c:\Users\Sealion\anaconda3\envs\medical-ai\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Sealion\anaconda3\envs\medical-ai\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called: get_claim_details
Arguments: {'claim_id': 'C1005'}


c:\Users\Sealion\anaconda3\envs\medical-ai\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Sealion\anaconda3\envs\medical-ai\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called: get_provider_risk
Arguments: {'provider_id': 'PR002'}


c:\Users\Sealion\anaconda3\envs\medical-ai\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## 15. Example questions

Try these in the chatbot:

```text
What is claim C1002?
```

```text
Is claim C1005 suspicious?
```

```text
What is the risk profile for provider PR002?
```

```text
Show me 3 anomalous claims.
```

```text
What is the utilization profile for patient P001?
```

```text
Can you explain why provider PR005 may need payment integrity review?
```